# Session 06: Confidence intervals in regression

Use this notebook while working through **Confidence intervals in regression**.

In **Simple linear regression: the line of best fit**, you drew the study-hours chart with `ci=None`. This notebook restores the confidence band and adds an interval for the slope.

A regression line is built from a sample, so a different sample would usually produce a slightly different line. The shaded band shows uncertainty around the estimated **mean response**. The slope interval helps answer a question left open in **Simple linear regression: the line of best fit**: does the sample provide evidence of a linear relationship in the population?

We return to the student habits data used in **Correlation: strength, direction, and causation** and **Simple linear regression: the line of best fit**.

## Setup

The file contains 45 students, with weekly study hours, nightly sleep hours, daily screen time, attendance, and exam score.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")

DATA_FOLDER = Path("data")
HABITS_FILE = DATA_FOLDER / "student_habits.csv"

habits = pd.read_csv(HABITS_FILE)

print("Students:", len(habits))
habits.head()

## 1. The confidence band hidden by `ci=None`

In **Simple linear regression: the line of best fit**, you drew this chart with `ci=None`. Draw it again without that argument and seaborn gives you its default: the fitted line wrapped in a shaded **95% confidence band**.

In [ ]:
# Call sns.regplot on habits with x="study_hours" and y="exam_score".
# Do not pass ci=None; let seaborn draw its default confidence band.
# Add a title and axis labels, then call plt.show().


Read the band as a sequence of intervals along the fitted line. At each observed study-hours value, the line gives the estimated **mean exam score**, and the band shows the uncertainty around that mean.

Notice two features:

- the band is narrow — at its tightest it spans only about three marks
- most of the 45 dots sit outside it — the band represents uncertainty around the fitted line, not the range of student scores

## 2. The slope has an interval too

The slope in the study-hours chart is about **4.75 marks per extra hour of weekly study**. It is a point estimate from a sample, so the question introduced in **From a sample to a claim about a population** applies: *how far off could the sample estimate be?*

`stats.linregress` reports the slope's standard error. Build the interval with the structure introduced in **The 95% confidence interval and its width**:

> estimate ± critical value × standard error

In [ ]:
# Fit the regression with stats.linregress(x, y).
# Get the critical value from stats.t.ppf(0.975, len(habits) - 2).
# Calculate margin = critical value * fit.stderr.
# Print the slope, standard error, 95% interval, and p-value.


The slope is **4.75**, with a 95% interval of **[4.13, 5.36]**.

Apply the reporting rule from **Reading intervals honestly**: report the size in the client's units, with its uncertainty. Within the observed range, each additional hour of weekly study is associated with an average exam score about 4.1 to 5.4 marks higher.

Now compare the interval with zero. A population slope of zero represents no linear relationship, and zero is not in [4.13, 5.36]. Under the regression assumptions, this is strong evidence of a positive linear association between study hours and exam score. It does not show that extra study **caused** the higher scores.

Carry forward the fit checks from **Assessing a linear fit: residuals and R-squared**, the extrapolation warning from **Using a regression line for prediction**, and the confounding-variable and causation warnings from **Correlation: strength, direction, and causation**.

## 3. A relationship the data cannot confirm

Now ask the same question of **sleep**: do students who sleep more score higher?

Draw the chart first, with the confidence band, then calculate the slope interval.

In [ ]:
# Call sns.regplot with x="sleep_hours" and y="exam_score".
# Keep seaborn's default confidence band.
# Add a title and axis labels, then call plt.show().


### Activity — Compare the bands

Compare the sleep-hours band with the study-hours band. Which band is wider, and what does that tell you about uncertainty in the two fitted lines?

In [ ]:
# Repeat the slope-interval calculation for sleep_hours against exam_score.
# Print the slope, standard error, 95% interval, and p-value.


The sleep slope is **1.57**, but its interval runs from **−2.62 to +5.76**. The sample is compatible with a range of negative and positive population slopes. If the population slope were zero, a p-value of 0.45 means a sample slope at least this far from zero would not be unusual.

The careful logic from **Comparing groups by their intervals** still applies:

- an interval that **excludes zero** provides evidence of a linear relationship
- an interval that **contains zero** means the data does not establish one

The second statement is deliberately cautious. [−2.62, 5.76] does not prove that sleep is irrelevant. "No linear relationship detected" and "no relationship" are distinct findings; only the first is supported.

### Activity — Evidence or uncertainty?

For each result, state what you would tell the client.

1. Study hours against exam score: slope 4.75, 95% CI [4.13, 5.36].
2. Sleep hours against exam score: slope 1.57, 95% CI [−2.62, 5.76].
3. A colleague fits screen time against exam score, gets a negative slope with p = 0.03, and reports that screens harm results. What else do you want to know before agreeing? Remember the association-and-causation warning from **Correlation: strength, direction, and causation**.

**Your answer:** Double-click this cell and replace this text with your response.

## 4. The band is for the line, not the students

This is the same mean-versus-individual distinction explained in **The 95% confidence interval and its width**.

The band answers the question: *where is the mean exam score for students who study this many hours?* It does **not** answer: *what score will one student get?* The next cell compares the two widths.

In [ ]:
residuals = (
    habits["exam_score"]
    - (fit.intercept + fit.slope * habits["study_hours"])
)
residual_standard_deviation = np.sqrt(
    (residuals ** 2).sum() / (len(habits) - 2)
)

standard_error_at_centre = residual_standard_deviation / np.sqrt(len(habits))
band_width_at_centre = (
    2
    * stats.t.ppf(0.975, len(habits) - 2)
    * standard_error_at_centre
)

print("Residual standard deviation:",
      round(residual_standard_deviation, 1), "marks")
print("Width of the band at the centre:",
      round(band_width_at_centre, 1), "marks in total")

The residual standard deviation is about **5.2 marks**, while the confidence band at its tightest spans about **3.1 marks in total**. That is why most dots lie outside the band: it was never intended to contain individual students.

The distinction matters:

- students with one additional weekly study hour averaged about 4 to 5 more marks — supported as an association within the observed range
- "Study one more hour and your score will rise five marks" — not supported

A prediction interval for one student must include individual variation and is much wider than the confidence band for the mean. This is the same distinction demonstrated with boarding counts in **The 95% confidence interval and its width**.

### Activity — Write the finding

Write two or three sentences reporting the study-hours result for a student services newsletter. Include the size in marks, its uncertainty, what the result does not say about an individual student, and the association-and-causation caveat from **Correlation: strength, direction, and causation**.

**Your answer:** Double-click this cell and replace this text with your response.

## What you have done

- Restored the confidence band hidden by `ci=None` and read it as uncertainty around the estimated mean response.
- Built the study-hours slope interval: 4.75 marks per hour, with a 95% interval of [4.13, 5.36].
- Used zero as the reference point and found strong evidence of a positive linear association for study hours.
- Examined the sleep-hours interval [−2.62, 5.76], which contains zero, so the sample does not establish a linear relationship.
- Separated uncertainty about the mean response from variation between individual students.
- Reported associations without turning them into causal claims.

The reporting habits from **Reading intervals honestly** apply to slopes just as they do to means: report the size, include its uncertainty, and state what the data cannot support.